# 01 — Data validation and cleaning

Review the real local AskReddit dataset and the reproducible PySpark cleaner.
This notebook performs data quality validation and cleaning only. No downloads,
installations, labels, predictive features, advanced EDA, or modeling.

Start Jupyter with Java 21 selected and use the project `.venv` kernel. The default
reviews an existing completed run. To create a new run, set `RUN_PIPELINE = True`
and choose **new** clean, quarantine, and report paths. Original Parquet files are
never modified. See `docs/data-validation-cleaning.md` for the full policies.

In [ ]:
from pathlib import Path
import json
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/askreddit/clean.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the project repository")
sys.path.insert(0, str(ROOT / "src"))
from askreddit.clean import Policy, create_spark, resolve_inputs, run_cleaning, utc_seconds
from askreddit.quality import verify_cleaned

RUN_PIPELINE = False
INPUTS = [str(ROOT / "data/extracted/askreddit-2025-??.parquet")]
CLEAN = ROOT / "data/processed/askreddit_clean"
QUARANTINE = ROOT / "data/processed/askreddit_quarantine"
REPORTS = ROOT / "reports/data_quality"
POLICY = Policy(min_created_utc=utc_seconds("2005-01-01T00:00:00Z"),
                max_created_utc=utc_seconds("2026-11-19T00:00:00Z"),
                timestamp_unit="seconds")
print("Python:", sys.version.split()[0])
print("Selected input files:", len(resolve_inputs(INPUTS)))

## Optional pipeline execution

All rules come from the reusable package. The CLI and notebook use the same path
guards, schema checks, quarantine policy, and read-back invariants. The pipeline
refuses existing destinations. It reports all explicitly supplied corrupt files.

In [ ]:
if RUN_PIPELINE:
    spark = create_spark()
    try:
        result = run_cleaning(spark, resolve_inputs(INPUTS), CLEAN, QUARANTINE, REPORTS, POLICY)
        print("Pipeline status:", result["status"])
    finally:
        spark.stop()
else:
    print("Reviewing saved outputs; optional pipeline execution is disabled.")

In [ ]:
def read_json(relative):
    path = REPORTS / relative
    if not path.is_file():
        raise FileNotFoundError(f"Missing completed-run report: {path}. Run the CLI first.")
    return json.loads(path.read_text())

run = read_json("run.json")
if run["status"] != "completed":
    raise RuntimeError("This notebook requires a completed cleaning run")
before = read_json("before_cleaning/profile.json")
after = read_json("after_cleaning/profile.json")
schema = read_json("before_cleaning/schema.json")
print(json.dumps({
    "input_files": schema["file_count"],
    "distinct_schemas": schema["distinct_schema_count"],
    "input_records": before["total_records"],
    "coverage_utc": [before["min_created_utc_iso"], before["max_created_utc_iso"]],
    "nulls_before": before["nulls"],
    "raw_duplicates": before["duplicates"],
    "cleaned_records": after["total_cleaned_records"],
    "quarantined_records": after["total_quarantined_records"],
    "duplicate_losers": after["total_duplicate_records_removed"],
    "primary_rejection_counts": after["primary_rejection_counts"],
    "retained_percentage": after["retained_percentage"],
    "invariants": after["invariants"],
}, indent=2))

In [ ]:
print("Monthly counts before cleaning (UTC):")
print(json.dumps(before["monthly_records"], indent=2))
print("Monthly counts after cleaning (UTC):")
print(json.dumps(after["monthly_records"], indent=2))
assert all(after["invariants"].values())
assert before["total_records"] == after["total_cleaned_records"] + after["total_quarantined_records"]

## Verify the saved Parquet datasets

This cell recomputes the post-cleaning invariants from the full saved datasets
using Spark. Only aggregate summaries reach the driver. The quarantine preserves
all rejected originals; multiple reasons may overlap, while primary reason counts
reconcile exactly. No original rows are displayed or collected into Pandas.

In [ ]:
spark = create_spark()
try:
    clean_frame = spark.read.parquet(str(CLEAN))
    quarantine_frame = spark.read.parquet(str(QUARANTINE))
    clean_frame.printSchema()
    checked = verify_cleaned(clean_frame, quarantine_frame, before["total_records"], Policy(**run["policy"]))
    assert checked["total_cleaned_records"] == after["total_cleaned_records"]
    assert checked["total_quarantined_records"] == after["total_quarantined_records"]
    print(json.dumps(checked["invariants"], indent=2))
finally:
    spark.stop()

## Audit interpretation

The original input hashes and schemas are in `before_cleaning/schema.json`.
Duplicate IDs and per-field variant counts are saved as distributed Parquet under
`before_cleaning/duplicate_ids/`. Quarantine includes raw values, original JSON,
source file, reasons, and the retained-record fingerprint for duplicate losers.

Scores equal to zero or below zero are preserved. Scores remain reserved for
future target construction and must never be predictive inputs. Review any
conflicting archive snapshots before choosing a different deduplication policy.
The historical sample, older pilot, and two zero-byte temporary inputs are listed
in `reports/repository_inspection.json` and have not been modified.